In [ ]:
import torch
import torch.nn.functional as F
from jaxtyping import Int

# custom utils
from muutils.misc import shorten_numerical_to_str
from trnbl import TrainingManager
from trnbl.loggers.local import LocalLogger


from attention_motifs.ae import AttnAEConfig, AttnAE
from attention_motifs.dataset.dataset import CollectedAttentionPatternDataloader
from attention_motifs.dataset.util import AttentionPatternMetadata

In [ ]:
# magic autoreload
%load_ext autoreload
%autoreload 2

In [ ]:
train_loader_dataset = CollectedAttentionPatternDataloader.read(
	"../data/activations/pile_5"
)
val_loader_dataset = CollectedAttentionPatternDataloader.read(
	"../data/activations/pile_5_val"
)

batch_size: int = 32
train_loader = train_loader_dataset.dataloader(batch_size)
val_loader = val_loader_dataset.dataloader(batch_size)

print(f"Train loader: {len(train_loader)} batches, {len(train_loader.dataset)} samples")

In [ ]:
config: AttnAEConfig = AttnAEConfig(
	latent_dim=64,
)

model: AttnAE = AttnAE(config)

model_n_params: int = sum(p.numel() for p in model.parameters())
print(
	f"model has {model_n_params} ({shorten_numerical_to_str(model_n_params)}) parameters"
)

# model

In [ ]:
train_loader: torch.utils.data.DataLoader
val_loader: torch.utils.data.DataLoader | None = None
num_epochs: int = 100
learning_rate: float = 1e-3
recon_weight: float = 1.0
contrast_weight: float = 1.0
project_name: str = "contrastive-ae"
checkpoint_interval: str = "1/10 run"
eval_interval: str = "1K samples"
device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = model.to(device)
optimizer: torch.optim.Optimizer = model.config.optimizer(
	model.parameters(),
	lr=learning_rate,
)

In [ ]:
def evaluation_step(model: AttnAE) -> dict[str, float]:
	"""Evaluate model on validation set"""
	if val_loader is None:
		return {}

	model.eval()
	val_metrics = {"val/loss": 0.0, "val/recon_loss": 0.0, "val/contrast_loss": 0.0}

	with torch.no_grad():
		for patterns, metadata in tr.batch_loop(train_loader):
			patterns = patterns.to(device).to(torch.float32).unsqueeze(1)
			optimizer.zero_grad()
			x_recon, embeddings = model(patterns)

			# reconstruction loss
			recon_loss = F.mse_loss(x_recon, patterns)

			# contrastive loss using all pairs in batch
			# compute "classes" for contrastive loss
			# classes is a tensor of the same shape as the batch, where each element is an integer
			classes: Int[torch.Tensor, " batch"] = (
				AttentionPatternMetadata.contrastive_classes(metadata).to(device)
			)

			# compute contrastive loss
			contrast_loss = model.contrastive_loss(embeddings, classes)

			# combined loss and backward pass
			total_loss = recon_weight * recon_loss + contrast_weight * contrast_loss
			total_loss.backward()
			optimizer.step()

			val_metrics["val/loss"] += total_loss.item()
			val_metrics["val/recon_loss"] += recon_loss.item()
			val_metrics["val/contrast_loss"] += contrast_loss.item()

	for k in val_metrics:
		val_metrics[k] /= len(val_loader)

	model.train()
	return val_metrics

In [ ]:
# setup logger
logger: LocalLogger = LocalLogger(
	project=project_name,
	metric_names=[
		"train/loss",
		"train/recon_loss",
		"train/contrast_loss",
		"val/loss",
		"val/recon_loss",
		"val/contrast_loss",
	],
	train_config=dict(
		model_config=model.zanj_model_config.serialize(),
		learning_rate=learning_rate,
		recon_weight=recon_weight,
		contrast_weight=contrast_weight,
	),
)

with TrainingManager(
	model=model,
	logger=logger,
	evals={
		eval_interval: evaluation_step,
	}.items(),
	checkpoint_interval=checkpoint_interval,
) as tr:
	for epoch in tr.epoch_loop(range(num_epochs)):
		for patterns, metadata in tr.batch_loop(train_loader):
			patterns = patterns.to(device).to(torch.float32).unsqueeze(1)
			optimizer.zero_grad()
			x_recon, embeddings = model(patterns)

			# reconstruction loss
			recon_loss = F.mse_loss(x_recon, patterns)

			# contrastive loss using all pairs in batch
			batch_size = patterns.size(0)

			# compute "classes" for contrastive loss
			# classes is a tensor of the same shape as the batch, where each element is an integer
			classes: Int[torch.Tensor, " batch"] = (
				AttentionPatternMetadata.contrastive_classes(metadata)
			)

			# compute contrastive loss
			contrast_loss = model.contrastive_loss(embeddings, classes)

			# combined loss and backward pass
			total_loss = recon_weight * recon_loss + contrast_weight * contrast_loss
			total_loss.backward()
			optimizer.step()

			# log metrics
			tr.batch_update(
				samples=len(metadata),
				**{
					"train/loss": total_loss.item(),
					"train/recon_loss": recon_loss.item(),
					"train/contrast_loss": contrast_loss.item(),
				},
			)

			del patterns, x_recon, embeddings, recon_loss, contrast_loss, total_loss